In [1]:
import sqlite3
import shutil
import pandas as pd

# Define database paths
original_db_path = "../Datasets/database/mfa.db"
copied_db_path = "Outputs/mfa.db"

# Make a copy of the original database
shutil.copy(original_db_path, copied_db_path)
print("Database copied to 'Outputs/mfa.db'.")

# Connect to the copied database
conn = sqlite3.connect(copied_db_path)
cursor = conn.cursor()


Database copied to 'Outputs/mfa.db'.


In [ ]:
# Inserting

-- Demonstrates adding individual rows to a table
-- Uses mfa.db

-- Adds a new item to the collections
INSERT INTO "collections" ("id", "title", "accession_number", "acquired")
VALUES (1, 'Profusion of flowers', '56.257', '1956-04-12');

-- Adds a new item to the collections
INSERT INTO "collections" ("id", "title", "accession_number", "acquired")
VALUES (2, 'Farmers working at dawn', '11.6152', '1911-08-03');

-- Adds a new item to the collections, demonstrating primary key auto-increments
INSERT INTO "collections" ("title", "accession_number", "acquired")
VALUES ('Spring outing', '14.76', '1914-01-08');

-- Shows violation of UNIQUE
INSERT INTO "collections" ("title", "accession_number", "acquired")
VALUES ('Spring outing', '14.76', '1914-01-08');

-- Shows violation of NOT NULL
INSERT INTO "collections" ("title", "accession_number", "acquired")
VALUES (NULL, '56.496', '1914-01-08');

-- Profusion of flowers: https://collections.mfa.org/objects/254/profusion-of-flowers?ctx=59408041-a021-4b91-bceb-580fd6fe7e17&idx=5
-- Farmers working at dawn: https://collections.mfa.org/objects/256/farmers-working-at-dawn?ctx=59408041-a021-4b91-bceb-580fd6fe7e17&idx=7
-- Spring outing: https://collections.mfa.org/objects/353/spring-outing?ctx=87931f50-caf4-4309-8175-96c5196e52bb&idx=23

-- Demonstrates adding multiple rows to a table
-- Uses mfa.db

-- Adds a set of new items to the collection
INSERT INTO "collections" ("title", "accession_number", "acquired") 
VALUES 
('Imaginative landscape', '56.496', NULL),
('Peonies and butterfly', '06.1899', '1906-01-01');

-- Imaginative landscape: https://collections.mfa.org/objects/318/imaginative-landscape?ctx=792f664e-1945-4e0e-9c34-1de7d932bf36&idx=14
-- Peonies and butterfly: https://collections.mfa.org/objects/8532/peonies-and-butterfly?ctx=26ebba4c-6b5c-4234-8e01-60eba690cd70&idx=65


In [ ]:
# Updating

-- Demonstrates updating authorship
-- Uses mfa.db

-- Updates authorship (incorrectly)
UPDATE "created" SET "artist_id" = (
    SELECT "id" FROM "artists"
    WHERE "name" = 'Li Yin'
);

-- Updates authorship (correctly) for a piece with a previously unknown authorship
UPDATE "created" SET "artist_id" = (
    SELECT "id" FROM "artists"
    WHERE "name" = 'Li Yin'
)
WHERE "collection_id" = (
    SELECT "id" FROM "collections"
    WHERE "title" = 'Farmers working at dawn'
);


-- Demonstrates cleaning data from a CSV of votes for favorite artwork
-- Creates votes.db

-- Imports votes.csv
.import votes.csv votes

-- Counts votes
SELECT "title", COUNT("title") FROM "votes" GROUP BY "title";

-- Removes trailing whitespace
UPDATE "votes" SET "title" = trim("title");

-- Forces to uppercase
UPDATE "votes" SET "title" = upper("title");

-- Manually updates the titles of "Farmers working at dawn"
UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" = 'FARMERS WORKING';

UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" = 'FAMERS WORKING AT DAWN';

-- Fixes misspellings of "Farmers working at dawn"
UPDATE "votes" SET "title" = 'FARMERS WORKING AT DAWN'
WHERE "title" LIKE 'Fa%';

-- Fixes misspellings of "Imaginative landscape"
UPDATE "votes" SET "title" = 'IMAGINATIVE LANDSCAPE'
WHERE "title" LIKE 'Imag%';

-- Fixes misspellings of "Profusion of flowers"
UPDATE "votes" SET "title" = 'PROFUSION OF FLOWERS'
WHERE "title" LIKE 'Profusion %';


In [ ]:
# Deleting

-- Demonstrates deleting rows from a single table
-- Uses mfa.db

-- Deletes item with particular title
DELETE FROM "collections" WHERE "title" = 'Spring outing';

-- Deletes item where value is NULL
DELETE FROM "collections" WHERE "acquired" IS NULL;

-- Deletes items acquired before the museum moved to a new location in 1909
DELETE FROM "collections" WHERE "acquired" < '1909-01-01';

-- Demonstrates deleting rows with constraints
-- Uses mfa.db

-- Raises a foreign key constraint error
DELETE FROM "artists" WHERE "name" = 'Unidentified artist';

-- Deletes the artist's affiliation with their work, using hard-coded id
DELETE FROM "created" WHERE "artist_id" = 3;

-- Deletes the artist's affiliation with their work, using subquery
DELETE FROM "created" WHERE "artist_id" = (
    SELECT "id" FROM "artists" WHERE "name" = 'Unidentified artist'
);

-- Deletes the artist themselves
DELETE FROM "artists" WHERE "name" = 'Unidentified artist';


In [ ]:
# Triggers

-- Demonstrates triggers on delete and insert
-- Uses mfa.db

-- Creates a table to track buying and selling of items from collections
CREATE TABLE "transactions" (
    "id" INTEGER,
    "title" TEXT,
    "action" TEXT,
    PRIMARY KEY("id")
);

-- Creates a trigger to log selling items from collections
CREATE TRIGGER "sell" 
BEFORE DELETE ON "collections"
BEGIN
    INSERT INTO "transactions" ("title", "action")
    VALUES (OLD."title", 'sold');
END;

-- Lists existing triggers
.schema

-- Deletes from collections
DELETE FROM "collections" WHERE "title" = 'Profusion of flowers';

-- Creates a trigger to log buying items
CREATE TRIGGER "buy" 
AFTER INSERT ON "collections"
BEGIN
    INSERT INTO "transactions" ("title", "action")
    VALUES (NEW."title", 'bought');
END;

-- Adds item to collections
INSERT INTO "collections" ("title", "accession_number", "acquired")
VALUES ('Profusion of flowers', '56.257', '1956-04-12');


In [ ]:
# Soft delete



-- Demonstrates soft deletes
-- Uses mfa.db

-- Adds a "deleted" column to "collections" table
ALTER TABLE "collections" ADD COLUMN "deleted" INTEGER DEFAULT 0;

-- Views updated schema of collections table
.schema "collections"

-- Views data
SELECT * FROM "collections";

-- Instead of deleting an item, updates its deleted column to be 1
UPDATE "collections" SET "deleted" = 1 WHERE "title" = 'Farmers working at dawn';

-- Selects all items from collections that are not deleted
SELECT * FROM "collections" WHERE "deleted" != 1;



-- Creates a view to show only items in collections that are NOT deleted
CREATE VIEW "current_collections" AS
SELECT "id", "title", "accession_number", "acquired" FROM "collections" WHERE "deleted" = 0;

-- Selects from "current_collections" view to see non-deleted items
SELECT * FROM "current_collections";

-- Fails to delete an item from the view
DELETE FROM "current_collections" WHERE "title" = 'Imaginative landscape';

-- Creates trigger to delete items from a view
CREATE TRIGGER "delete"
INSTEAD OF DELETE ON "current_collections"
FOR EACH ROW
BEGIN
    UPDATE "collections" SET "deleted" = 1 WHERE "id" = OLD."id";
END;

-- Creates trigger to revert an item's deletion
CREATE TRIGGER "insert_when_exists"
INSTEAD OF INSERT ON "current_collections"
FOR EACH ROW 
WHEN NEW."accession_number" IN (SELECT "accession_number" FROM "collections")
BEGIN
    UPDATE "collections" SET "deleted" = 0 WHERE "accession_number" = NEW."accession_number";
END;

-- Creates trigger to insert a new item into collections
CREATE TRIGGER "insert_when_new"
INSTEAD OF INSERT ON "current_collections"
FOR EACH ROW
WHEN NEW."accession_number" NOT IN (SELECT "accession_number" FROM "collections")
BEGIN
    INSERT INTO "collections" ("title", "accession_number", "acquired")
    VALUES (NEW."title", NEW."accession_number", NEW."acquired");
END;
